In [1]:
import sys
sys.path.append('.venv/Lib/site-packages')

In [31]:
import pyautogui
import pandas as pd

df = pd.DataFrame(columns=['num', 'top', 'left', 'width', 'height'])

img_path = './imgs/'

ll = 0
for i in range(1, 9+1):
    positions = pyautogui.locateAllOnScreen(f'{img_path}{i}.png', confidence=0.90)
    for pos in positions:
      new_row = {'num':i, 'left':pos.left, 'top':pos.top, 'width':pos.width, 'height':pos.height}
      df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
      ll += 1

In [32]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
os.environ["OMP_NUM_THREADS"] = "2"

"""
n_clusters=170 >>> 170개가 될 때까지 반복하겠다.
"""
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=170)
kmeans.fit(df[['top', 'left']])
df['cluster'] = kmeans.labels_

# 같은 클러스터인 행끼리 평균을 해서 하나로 만듬
df = df.groupby(by='cluster').mean()

"""
left는 x좌표로 17개의 클러스터가 될 때까지
클러스터링하면 17개 열의 x좌표 위치가 반환됨

top의 경우 y좌표 10개, 10개의 행에 대한 y좌표 위치 반환
"""

kmeans = KMeans(n_clusters=17)
kmeans.fit(df[['left']])
df['x_cls'] = kmeans.labels_

kmeans = KMeans(n_clusters=10)
kmeans.fit(df[['top']])
df['y_cls'] = kmeans.labels_

"""
같은행이면 y좌표가 같아야 하고, 
같은열이면 x좌표가 같아야함
평균값으로 넣어줌
"""

for xi in range(17):
    mean_value = df.loc[df['x_cls'] == xi, 'left'].mean()  # 먼저 평균값 계산
    df.loc[df['x_cls'] == xi, 'left'] = mean_value  # loc를 사용하여 값 업데이트
for xi in range(10):
    mean_value = df.loc[df['y_cls'] == xi, 'top'].mean()  # 먼저 평균값 계산
    df.loc[df['y_cls'] == xi, 'top'] = mean_value  # loc를 사용하여 값 업데이트

import numpy as np
df = df.sort_values(by=['top', 'left']).reset_index(drop=True)
num1d = np.array(df['num'].values.tolist(), dtype=int)
num2d = num1d.reshape(10, 17)

In [33]:
"""
합이 10인 위치를 제거하는 기능을 10번 시행하기 위해 _로 열번 돌립니다.
특정 영역이 0이 되면서 없어지면 합이 0이 될 수 있는 영역이 새로 생길 수 있으므로
여러 번 돌려야 합니다.
"""
for _ in range(10):
    for yi in range(10):
        for xi in range(17):
            w_max, h_max = 17 - xi, 10 - yi
            for h in range(h_max):
                for w in range(w_max):
                    if np.sum(num2d[yi: yi + h+1, xi: xi + w+1]) == 10:
                        num2d[yi: yi + h+1, xi: xi + w+1] = 0

In [44]:
import pyautogui
import keyboard

"""
xloc, yloc의 평균 간격으로 사과 간 좌표의 간격을 구합니다.
dx는 x방향의 사과좌표 간격
dy는 y방향의 사과좌표 간격
"""
xloc = df['left'].unique()
yloc = df['top'].unique()

dx = np.mean(np.diff(xloc))
dy = np.mean(np.diff(yloc))

"""
좌표를 가져오는 이미지는 사과 안의 숫자 이미지입니다.
그래서 이 좌표의 영역으로 마우스 드래그를 하면 
드래그 영역에 사과가 완전히 속하지 않습니다.
그래서 사과가 드래그 영역에 들어올 수 있도록
-0.25 * dy, -0.25 * dy만큼 좌표 이동을 해야합니다.
"""
xloc_mouse = xloc - 0.25 * dx
yloc_mouse = yloc - 0.25 * dy

def mouse_move(xi, yi, w, h, duration):
	# 드래그 영역의 왼쪽, 위 좌표로 마우스 이동
    pyautogui.moveTo( xloc_mouse[xi], yloc_mouse[yi], duration=duration)
    # 마우스 누르는 상태
    pyautogui.mouseDown()
    # 드래그 영역의 오른쪽, 아래 좌표로 마우스 이동
    pyautogui.moveTo(xloc_mouse[xi] + (w+1)*dx, yloc_mouse[yi] + (h+1)*dy,  duration=duration)
    # 위의 좌표 이동과 똑같지만 이렇게 반복을 안하면 드래그가 제대로 안됩니다.
    # 어떻게 되는지 궁금하면 지우고 테스트해보세요.
    pyautogui.moveTo(xloc_mouse[xi] + (w+1)*dx, yloc_mouse[yi] + (h+1)*dy,  duration=duration)
    # 마우스 누르는 상태 해제
    pyautogui.mouseUp()

duration = 0.01
for _ in range(10):
    for yi in range(10):
        for xi in range(17):
        	# q를 꾹 누르고 있으면 취소되게 만듬
            if keyboard.is_pressed("q"):  
                break
            w_max, h_max = 17 - xi, 10 - yi
            for h in range(h_max):
                for w in range(w_max):
                    if np.sum(num2d[yi: yi + h+1, xi: xi + w+1]) == 10:
                        num2d[yi: yi + h+1, xi: xi + w+1] = 0
                        mouse_move(xi, yi, w, h, duration)

In [ ]:
import pyautogui
import pandas as pd
df = pd.DataFrame(columns=['num', 'top', 'left', 'width', 'height'])
img_path = './imgs/'

ll = 0
for i in range(1, 9+1):
    positions = pyautogui.locateAllOnScreen(f'{img_path}{i}.png', confidence=0.90)
    for pos in positions:
        new_row = {'num':i, 'top':pos.top, 'left':pos.left, 
                   'width':pos.width, 'height':pos.height}
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        ll += 1
        
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
os.environ["OMP_NUM_THREADS"] = "2"

from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=170)
kmeans.fit(df[['top', 'left']])
df['cluster'] = kmeans.labels_
df = df.groupby(by='cluster').mean()

kmeans = KMeans(n_clusters=17)
kmeans.fit(df[['left']])
df['x_cls'] = kmeans.labels_

kmeans = KMeans(n_clusters=10)
kmeans.fit(df[['top']])
df['y_cls'] = kmeans.labels_

for xi in range(17):
    mean_value = df.loc[df['x_cls'] == xi, 'left'].mean()  # 먼저 평균값 계산
    df.loc[df['x_cls'] == xi, 'left'] = mean_value  # loc를 사용하여 값 업데이트
for xi in range(10):
    mean_value = df.loc[df['y_cls'] == xi, 'top'].mean()  # 먼저 평균값 계산
    df.loc[df['y_cls'] == xi, 'top'] = mean_value  # loc를 사용하여 값 업데이트

import numpy as np
df = df.sort_values(by=['top', 'left']).reset_index(drop=True)
num1d = np.array(df['num'].values.tolist(), dtype=int)
num2d = num1d.reshape(10, 17)

import pyautogui
import keyboard
xloc = df['left'].unique()
yloc = df['top'].unique()

dx = np.mean(np.diff(xloc))
dy = np.mean(np.diff(yloc))
xloc_mouse = xloc - 0.25 * dx
yloc_mouse = yloc - 0.25 * dy

def mouse_move(xi, yi, w, h, duration):
    pyautogui.moveTo( xloc_mouse[xi], yloc_mouse[yi], duration=duration)
    pyautogui.mouseDown()
    pyautogui.moveTo(xloc_mouse[xi] + (w+1)*dx, yloc_mouse[yi] + (h+1)*dy,  duration=duration)
    pyautogui.moveTo(xloc_mouse[xi] + (w+1)*dx, yloc_mouse[yi] + (h+1)*dy,  duration=duration)
    pyautogui.mouseUp()

duration = 0.01
for _ in range(20):
    for yi in range(10):
        for xi in range(17):
            if keyboard.is_pressed("q"):  
                break
            w_max, h_max = 17 - xi, 10 - yi
            for h in range(h_max):
                for w in range(w_max):
                    if np.sum(num2d[yi: yi + h+1, xi: xi + w+1]) == 10:
                        num2d[yi: yi + h+1, xi: xi + w+1] = 0
                        mouse_move(xi, yi, w, h, duration)